In [0]:
%run ../00_utils

In [0]:
from pyspark.sql.window import Window

def build_dim_date(start="2015-01-01", end="2027-12-31"):
    first_policy = spark.table("workspace.gold.ref_policy_rate").agg(F.min("effective_date")).collect()[0][0]
    days = spark.sql(f"SELECT explode(sequence(DATE'{first_policy}', DATE'{end}', INTERVAL 1 DAY)) AS full_date")

    pol = spark.table("workspace.gold.ref_policy_rate").select(
        F.col("effective_date").alias("full_date"), "policy_rate_pct", "rate_change_bps")
    w = Window.orderBy("full_date").rowsBetween(Window.unboundedPreceding, 0)
    days = (days.join(pol, "full_date", "left")
                .withColumn("is_policy_decision_day", F.col("policy_rate_pct").isNotNull())
                .withColumn("rate_change_bps", F.coalesce("rate_change_bps", F.lit(0)))
                .withColumn("policy_rate_pct", F.last("policy_rate_pct", ignorenulls=True).over(w))
                .where(F.col("full_date") >= F.lit(start)))

    trading = (spark.table("workspace.silver.daily_prices").select(F.col("trade_date").alias("full_date"))
                    .distinct().withColumn("is_trading_day", F.lit(True)))

    dim = (days.join(trading, "full_date", "left")
        .withColumn("is_trading_day", F.coalesce("is_trading_day", F.lit(False)))
        .select(F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"), "full_date",
                F.year("full_date").alias("year"), F.quarter("full_date").alias("quarter"),
                F.month("full_date").alias("month"), F.date_format("full_date", "MMMM").alias("month_name"),
                F.date_format("full_date", "yyyy-MM").alias("year_month"),
                F.weekofyear("full_date").alias("week_of_year"), F.date_format("full_date", "EEEE").alias("day_name"),
                F.dayofweek("full_date").isin(1, 7).alias("is_weekend"), "is_trading_day",
                "policy_rate_pct", "rate_change_bps", "is_policy_decision_day",
                F.current_timestamp().alias("load_timestamp")))

    dim.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_date")
    return dim.count()

In [0]:
def equities(df):
    return (df.where(F.col("sector_code").isNotNull())
              .where(~F.col("sector_code").isin("36", "3610"))
              .where(~F.col("symbol").endswith("PS")))

def build_dim_stock():
    s = equities(spark.table("workspace.silver.daily_prices"))

    latest = Window.partitionBy("symbol").orderBy(F.col("trade_date").desc())
    last_row = (s.withColumn("rn", F.row_number().over(latest)).where("rn = 1")
                 .select("symbol", "company_name", "sector_code"))

    stats = s.groupBy("symbol").agg(F.min("trade_date").alias("first_trade_date"),
                                    F.max("trade_date").alias("last_trade_date"),
                                    F.count("*").cast("int").alias("trading_days"))
    max_date = s.agg(F.max("trade_date")).collect()[0][0]
    ref = spark.table("workspace.gold.ref_sector").select("sector_code", "sector_name", "security_type")

    dim = (last_row.join(stats, "symbol").join(ref, "sector_code", "left")
        .select(F.xxhash64("symbol").alias("stock_key"), "symbol",
                F.trim(F.regexp_replace("company_name", r"(XDXB|XDXR|XD|XB|XR)$", "")).alias("company_name"),
                "sector_code",
                F.coalesce("sector_name", F.lit("Legacy sector (pre-2015 codes)")).alias("sector_name"),
                F.coalesce("security_type", F.lit("Legacy")).alias("security_type"),
                "first_trade_date", "last_trade_date", "trading_days",
                (F.col("last_trade_date") >= F.date_sub(F.lit(max_date), 30)).alias("is_active"),
                F.current_timestamp().alias("load_timestamp")))

    dim.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.dim_stock")
    return dim.count()

In [0]:
import math

def build_fact_daily_stock(start, end):
    # read 120 extra days so the first days of the range have full 20/60-day windows
    lookback = (dt.date.fromisoformat(start) - dt.timedelta(days=120)).isoformat()
    s = equities(spark.table("workspace.silver.daily_prices")).where(F.col("trade_date").between(lookback, end))

    # 1. daily return + rolling volatility (per stock, ordered by date)
    w20 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-19, 0)
    w60 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-59, 0)
    ann = math.sqrt(252)
    f = (s.withColumn("daily_return", F.when((F.col("close") > 0) & (F.col("ldcp") > 0),
                                             F.log(F.expr("try_divide(close, ldcp)"))))
          .withColumn("is_extreme_move", F.coalesce(F.abs("daily_return") > 0.15, F.lit(False)))
          .withColumn("ret_clean", F.when(F.col("is_traded") & ~F.col("is_extreme_move"), F.col("daily_return")))
          .withColumn("n20", F.count("ret_clean").over(w20))
          .withColumn("n60", F.count("ret_clean").over(w60))
          .withColumn("vol_20d", F.when(F.col("n20") >= 15, F.stddev("ret_clean").over(w20) * ann))
          .withColumn("vol_60d", F.when(F.col("n60") >= 45, F.stddev("ret_clean").over(w60) * ann))
          .withColumn("avg_volume_20d", F.avg("volume").over(w20))
          .where(F.col("trade_date").between(start, end)))

    # 2. risk percentile + tier (rank among all rated stocks on the same day)
    wd = Window.partitionBy("trade_date")
    wr = Window.partitionBy("trade_date").orderBy(F.col("vol_20d").asc_nulls_last())
    f = (f.withColumn("n_rated", F.count("vol_20d").over(wd))
          .withColumn("rk", F.rank().over(wr))
          .withColumn("risk_percentile", F.when(F.col("vol_20d").isNotNull(),
                                                F.expr("try_divide(rk - 1, n_rated - 1)")))
          .withColumn("risk_tier", F.when(F.col("risk_percentile").isNull(), "Not Rated")
                                    .when(F.col("risk_percentile") >= 0.8, "High")
                                    .when(F.col("risk_percentile") >= 0.4, "Medium")
                                    .otherwise("Low")))

    # 3. final columns + MERGE
    out = f.select(F.date_format("trade_date", "yyyyMMdd").cast("int").alias("date_key"),
                   F.xxhash64("symbol").alias("stock_key"),
                   "open", "high", "low", "close", "ldcp", "volume", "daily_return", "is_extreme_move",
                   "vol_20d", "vol_60d", "avg_volume_20d", "risk_percentile", "risk_tier",
                   "is_traded", "is_ohlc_valid", F.current_timestamp().alias("load_timestamp"))

    spark.sql("""CREATE TABLE IF NOT EXISTS workspace.gold.fact_daily_stock (
        date_key INT, stock_key BIGINT, open DOUBLE, high DOUBLE, low DOUBLE, close DOUBLE, ldcp DOUBLE,
        volume BIGINT, daily_return DOUBLE, is_extreme_move BOOLEAN, vol_20d DOUBLE, vol_60d DOUBLE,
        avg_volume_20d DOUBLE, risk_percentile DOUBLE, risk_tier STRING, is_traded BOOLEAN,
        is_ohlc_valid BOOLEAN, load_timestamp TIMESTAMP) CLUSTER BY (date_key, stock_key)""")

    out.createOrReplaceTempView("gold_stock_src")
    m = spark.sql("""MERGE INTO workspace.gold.fact_daily_stock t USING gold_stock_src s
        ON t.date_key = s.date_key AND t.stock_key = s.stock_key
        WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""").collect()[0]
    return m["num_inserted_rows"], m["num_updated_rows"]

In [0]:
def build_fact_market_daily(start, end):
    lookback = (dt.date.fromisoformat(start) - dt.timedelta(days=60)).isoformat()
    lk, sk, ek = [int(x.replace("-", "")) for x in (lookback, start, end)]
    b = spark.table("workspace.gold.fact_daily_stock").where(F.col("date_key").between(lk, ek))

    ok = F.col("is_traded") & ~F.col("is_extreme_move")
    agg = b.groupBy("date_key").agg(
        F.sum(F.col("is_traded").cast("int")).cast("int").alias("stocks_traded"),
        F.sum(F.when(ok & (F.col("daily_return") > 0), 1).otherwise(0)).cast("int").alias("advancers"),
        F.sum(F.when(ok & (F.col("daily_return") < 0), 1).otherwise(0)).cast("int").alias("decliners"),
        F.sum("volume").cast("bigint").alias("total_volume"),
        F.avg(F.when(ok, F.col("daily_return"))).alias("market_return"),
        F.percentile_approx("vol_20d", 0.5).alias("median_stock_vol_20d"),
        F.expr("try_divide(sum(CASE WHEN risk_tier = 'High' THEN 1 ELSE 0 END), "
               "sum(CASE WHEN risk_tier <> 'Not Rated' THEN 1 ELSE 0 END))").alias("pct_high_risk"))

    wm = Window.orderBy("date_key").rowsBetween(-19, 0)
    agg = (agg.withColumn("n", F.count("market_return").over(wm))
              .withColumn("market_vol_20d",
                          F.when(F.col("n") >= 15, F.stddev("market_return").over(wm) * math.sqrt(252)))
              .where(F.col("date_key").between(sk, ek))
              .select("date_key", "stocks_traded", "advancers", "decliners", "total_volume", "market_return",
                      "market_vol_20d", "median_stock_vol_20d", "pct_high_risk",
                      F.current_timestamp().alias("load_timestamp")))

    spark.sql("""CREATE TABLE IF NOT EXISTS workspace.gold.fact_market_daily (
        date_key INT, stocks_traded INT, advancers INT, decliners INT, total_volume BIGINT,
        market_return DOUBLE, market_vol_20d DOUBLE, median_stock_vol_20d DOUBLE, pct_high_risk DOUBLE,
        load_timestamp TIMESTAMP) CLUSTER BY (date_key)""")

    agg.createOrReplaceTempView("gold_market_src")
    m = spark.sql("""MERGE INTO workspace.gold.fact_market_daily t USING gold_market_src s
        ON t.date_key = s.date_key
        WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""").collect()[0]
    return m["num_inserted_rows"], m["num_updated_rows"]

In [0]:
def run_step(batch_id, load_type, parameter, fn, *args):
    t0 = dt.datetime.now()
    try:
        res = fn(*args)
        ins, upd = (res, 0) if isinstance(res, int) else res
        write_log(batch_id, "silver_to_gold", load_type, parameter, t0, dt.datetime.now(), "SUCCESS",
                  rows_inserted=ins, rows_updated=upd)
        print(f"  {parameter}: rows={ins}, updated={upd}, {(dt.datetime.now() - t0).seconds}s")
        return True
    except Exception as e:
        write_log(batch_id, "silver_to_gold", load_type, parameter, t0, dt.datetime.now(), "FAILURE",
                  error_message=str(e)[:1000])
        print("  FAILED:", parameter, str(e)[:300])
        return False

def load_gold(start_date, end_date, load_type):
    if load_type == "incremental":
        last = spark.sql("SELECT MAX(date_key) FROM workspace.gold.fact_daily_stock").collect()[0][0]
        newest = spark.sql("SELECT MAX(trade_date) FROM workspace.silver.daily_prices").collect()[0][0]
        start_date = (dt.datetime.strptime(str(last), "%Y%m%d").date() + dt.timedelta(days=1)).isoformat()
        end_date = newest.isoformat()
        print(f"Incremental: processing {start_date} to {end_date}")

    batch_id = f"gold_{dt.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
    rng = f"{start_date}..{end_date}"
    ok = run_step(batch_id, load_type, "dim_date", build_dim_date)
    ok = ok and run_step(batch_id, load_type, "dim_stock", build_dim_stock)
    ok = ok and run_step(batch_id, load_type, f"fact_daily_stock {rng}", build_fact_daily_stock, start_date, end_date)
    ok = ok and run_step(batch_id, load_type, f"fact_market_daily {rng}", build_fact_market_daily, start_date, end_date)
    print(f"Done. batch={batch_id}, status={'SUCCESS' if ok else 'FAILURE'}")
    return batch_id